In [ ]:
%load_ext autoreload
%autoreload 2
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import os

import calibration as cal                      # the calibration steps (also runnable offline: run_offline.sh)
from utils import calib_dir, result_dir, paths

In [ ]:
import matplotlib as mpl
from cycler import cycler

cmap = mpl.colormaps['hot'] # 1. Choose your colormap (e.g., 'hot', 'jet', 'turbo')
colors = cmap(np.linspace(0.0, 1.0, 7)) # 2. Extract a set of colors from it (e.g., 6 distinct colors across the map)
# plt.rcParams['axes.prop_cycle'] = cycler(color=colors) # 3. Apply it to the default line cycle
plt.rcParams['image.cmap'] = 'magma' # 4. Change the default colormap for images/scatter plots
plt.rcParams.update({
    "text.usetex": True, # Use LaTeX to write all text
    "font.family": "serif", # Use a serif font
    "font.serif": ["Computer Modern Roman"] # Specifically use the classic LaTeX font
})
plt.rcParams['image.origin'] = 'lower'

Calibration of the e2e system in the **aoeb layout** (`calibration.py`, names from `aoeb/naming.py`), so that the budget machine reads
the products (`budget_e2e.yml`, `cal.machine()`). Every step skips what is already on disk. Both pyramids are simulated in every
aliasing / PC run (same rMod, VIS at `wavelengths['vis']`, NIR at `wavelengths['nir']`); any other sensing wavelength is derived from
the nearest of these (aoeb ASSUMPTIONS.md, A37). The long steps run offline: `bash run_offline.sh "alias pc" "0 1" --seeings 0.9`.

In [ ]:
cfg = cal.E2EConfig(
    calib_dir=calib_dir,
    n_subaps=(10, 14, 24, 36, 48),
    rMods=(0.0, 0.5, 1.0, 2.0, 3.0),
    modes_per_subap={10: [54], 14: [54, 100], 24: [54, 100, 300], 36: [54, 100, 300, 800], 48: [54, 100, 300, 800, 1300]},
    wavelengths={'vis': 900.0, 'nir': 1300.0},
    seeings=(0.9,),
    n_screens=70,                       # PC IMs: push-pull cycles, one screen each
)
print(cal.status(cfg))

In [ ]:
# Pupil data
cal.calibrate_pupdata(cfg)

In [ ]:
# Diffraction-limited IMs (pyr_vis at every sensing wavelength) and SVD reconstructors
cal.calibrate_im(cfg)
cal.compute_recs(cfg)

In [ ]:
# Aliasing runs (independent screens every frame, N modes removed exactly): aliasing, average frame, slope null, throughput of both
# pyramids. The frame rate does not change these snapshot products (no rate tag); the budget machine applies the exposure average and
# the loop rejection at the frame rate it is asked for (utils.get_aliasing_power(..., fs=...) does the same for one frame).
cal.calibrate_alias(cfg)

In [ ]:
# PC IMs (optical gains): one run per case, n_screens push-pull cycles. Long: ~33 GPU-hours per seeing for all the cases (48x48 with
# 1300 modes: ~2.4 h per case) -> better offline: bash run_offline.sh pc "1 1 0" --seeings 0.9
cal.calibrate_pc(cfg)

In [ ]:
# ML reconstructors of the unmodulated pyramid (aoeb: full sx-sy noise covariance), noise weights of the average frame
cal.compute_ml(cfg, rMods=(0.0,))

In [ ]:
# The budget machine on these products (any sensing wavelength: the missing ones are derived from 900 / 1300 nm)
m = cal.machine(cfg)
r = m.budget(rMod=3.0, n_subap=48, seeing=0.9, flux=1e8, fs=500., N=1300, wavelength_nm=900.)
r.totals()